# Data Visualisation

This notebook will contain the final project visualisations.

# Setup

In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_project_root(start_path: Path) -> Path:
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Project root not found. Run this notebook inside the repository."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# Imports & Visualization Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter

from src.data_loader import load_data
from src.analysis.orders import build_order_analysis_view
from src.analysis.summary import summarize_delivery_by_slot

from src.analysis.investigation import (
    investigate_delivery_stages,
    classify_order_routes,
    summarize_routes,
    build_focus_cohort,
    summarize_state_pairs,
    summarize_focus_time,
)

# Output directory
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "revised_charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Consistent color palette
COLORS = {
    "orange": "#F8B88B",
    "blue": "#9CC3E6",
    "green": "#A8D5BA",
    "green_line": "#6DBE8C",
    "blue_line": "#7FAFE0",
    "reference": "#555555",
    "text": "#333333",
    "secondary": "#666666",
    "highlight": "#FBE5D3",
}

SLOTS = [
    "00:00–06:00",
    "06:00–12:00",
    "12:00–18:00",
    "18:00–24:00",
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "text.color": COLORS["text"],
    "axes.labelcolor": COLORS["text"],
    "axes.titlecolor": COLORS["text"],
    "xtick.color": COLORS["text"],
    "ytick.color": COLORS["text"],
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
    "savefig.dpi": 200,
})

print(f"Figures will be saved to: {OUTPUT_DIR}")

# Load & Build Order-Level Dataset

In [ ]:
# Load preprocessed item-level dataset
cleaned_items = load_data(
    "cleaned_order_items",
    stage="processed"
)

# Build order-level analysis view
orders, order_diagnostics = build_order_analysis_view(
    cleaned_items
)

print("Dataset loaded successfully!")
print(f"Item-level records: {len(cleaned_items):,}")
print(f"Analyzable orders: {len(orders):,}")

display(order_diagnostics["cohort_summary"])

# Generate All Required Summaries

In [ ]:
# Figure 1: Delivery by approval-time slot
slot_summary = summarize_delivery_by_slot(orders)

# Figure 2: Route classification and slot summaries
stage_profiles = investigate_delivery_stages(orders)

order_routes = classify_order_routes(
    cleaned_items,
    orders
)

route_profiles = summarize_routes(
    orders,
    order_routes,
    stage_profiles["records"]
)

# Figure 3: Focus cohort and state pairs
focus_orders, focus_diagnostics = build_focus_cohort(
    cleaned_items,
    orders,
    order_routes
)

pair_profiles = summarize_state_pairs(
    cleaned_items,
    focus_orders
)

# Identify the leading pair, matching the original notebook
leading_state_pair = None

if not pair_profiles["pairs"].empty:
    leader = pair_profiles["pairs"].iloc[0]

    if leader["late_orders"] > 0:
        leading_state_pair = (
            leader["seller_state"],
            leader["customer_state"]
        )

# Figure 4: Monthly comparisons
focus_time = summarize_focus_time(
    pair_profiles["orders"],
    leading_state_pair
)

print("All summaries generated successfully!")
print(f"Leading state pair: {leading_state_pair}")

print("\nSlot summary:")
display(slot_summary)

print("\nRoute-slot summary:")
display(route_profiles["slots"])

print("\nTop 5 state pairs:")
display(pair_profiles["pairs"].head(5))

# Validate Data & Identify Columns

In [ ]:
# Validate order-level totals
assert slot_summary["orders"].sum() == len(orders), \
    "Slot order counts do not match the analysis cohort."

assert slot_summary["late_orders"].sum() == orders["is_late"].sum(), \
    "Late-order counts are inconsistent."

# Identify route classification column
route_slots = route_profiles["slots"].copy()

def normalize_route(value):
    return "".join(
        c for c in str(value).lower() if c.isalpha()
    )

route_col = next(
    (
        col for col in route_slots.columns
        if route_slots[col].map(normalize_route)
        .isin(["intrastate", "interstate"]).any()
    ),
    None
)

if route_col is None:
    raise ValueError(
        f"Route column not found: {route_slots.columns.tolist()}"
    )

route_slots["_route"] = route_slots[route_col].map(normalize_route)

# The proposed figures focus on SP -> RJ
assert leading_state_pair == ("SP", "RJ"), (
    f"Unexpected leading state pair: {leading_state_pair}"
)

# Overall reference rate
overall_late_rate = (
    100 * slot_summary["late_orders"].sum()
    / slot_summary["orders"].sum()
)

# Inter-state afternoon reference rate
inter_afternoon = route_slots.loc[
    route_slots["_route"].eq("interstate")
    & route_slots["approval_slot"].eq("12:00–18:00")
].iloc[0]

inter_reference_rate = (
    100 * inter_afternoon["late_orders"]
    / inter_afternoon["orders"]
)

print(f"Overall late rate: {overall_late_rate:.2f}%")
print(f"Inter-state afternoon rate: {inter_reference_rate:.2f}%")
print(f"Route column: {route_col}")
print("Data validation passed!")

# Figure 1: Late Count vs Late Rate

In [ ]:
def plot_revised_figure_1(slot_summary):

    data = (
        slot_summary
        .set_index("approval_slot")
        .reindex(SLOTS)
    )

    late_counts = data["late_orders"].to_numpy()
    rates = (
        100 * data["late_orders"] / data["orders"]
    ).to_numpy()

    overall = (
        100 * data["late_orders"].sum()
        / data["orders"].sum()
    )

    x = np.arange(len(SLOTS))

    fig, (ax_count, ax_rate) = plt.subplots(
        1, 2,
        figsize=(8.5, 3.9),
        gridspec_kw={"wspace": 0.28}
    )

    # Panel 1: Late count
    ax_count.bar(
        x, late_counts,
        width=0.62,
        color=COLORS["orange"]
    )

    ax_count.set_title("Late count", fontsize=10)
    ax_count.set_ylabel(
        "Late count (number of orders)", fontsize=9
    )
    ax_count.set_xlabel(
        "Approval time slot", fontsize=8.5
    )

    count_max = late_counts.max()
    ax_count.set_ylim(0, count_max * 1.2)

    ax_count.yaxis.set_major_formatter(
        FuncFormatter(lambda value, _: f"{value:,.0f}")
    )

    for i, value in enumerate(late_counts):
        ax_count.text(
            i, value + count_max * 0.015,
            f"{value:,.0f}",
            ha="center", va="bottom",
            fontsize=7.5
        )

    # Panel 2: Late rate
    ax_rate.bar(
        x, rates,
        width=0.62,
        color=COLORS["blue"]
    )

    ax_rate.axhline(
        overall,
        linestyle="--",
        linewidth=1,
        color="#4F7CAC",
        label=f"All orders: {overall:.2f}%"
    )

    ax_rate.set_title("Late rate", fontsize=10)
    ax_rate.set_ylabel(
        "Late deliveries (% of orders)", fontsize=9
    )
    ax_rate.set_xlabel(
        "Approval time slot", fontsize=8.5
    )
    ax_rate.set_ylim(0, 10)

    for i, rate in enumerate(rates):
        ax_rate.text(
            i, max(rate, overall) + 0.2,
            f"{rate:.2f}%",
            ha="center", va="bottom",
            fontsize=7.5
        )

    # Shared formatting
    for ax in (ax_count, ax_rate):
        ax.set_xticks(x)
        ax.set_xticklabels(SLOTS, fontsize=7.5)
        ax.tick_params(axis="y", labelsize=8)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    ax_rate.legend(
        loc="upper right",
        frameon=False,
        fontsize=7.5
    )

    fig.suptitle(
        "Figure 1. Late deliveries by approval-time slot",
        fontsize=11
    )

    fig.text(
        0.5, 0.01,
        "Left: late count (number of late orders). "
        "Right: late rate = late orders / total orders in the slot. "
        "Both vertical axes start at 0.",
        ha="center",
        fontsize=7
    )

    fig.subplots_adjust(
        bottom=0.2,
        top=0.84,
        left=0.09,
        right=0.985
    )

    path = OUTPUT_DIR / "figure_01_revised.png"
    fig.savefig(path, dpi=200)

    display(fig)
    plt.close(fig)

    print(f"Saved: {path}")


plot_revised_figure_1(slot_summary)

# Figure 2: Intra-state vs Inter-state

In [ ]:
def plot_revised_figure_2(route_slots):

    data = route_slots[
        route_slots["_route"].isin(
            ["intrastate", "interstate"]
        )
    ].copy()

    x = np.arange(len(SLOTS))
    width = 0.36

    fig, ax = plt.subplots(figsize=(8.5, 4.2))

    groups = [
        ("intrastate", "Intra-state",
         COLORS["blue"], -width / 2),

        ("interstate", "Inter-state",
         COLORS["orange"], width / 2),
    ]

    for route_key, label, color, offset in groups:

        subset = (
            data[data["_route"].eq(route_key)]
            .set_index("approval_slot")
            .reindex(SLOTS)
        )

        counts = subset["orders"].fillna(0).to_numpy()
        late = subset["late_orders"].fillna(0).to_numpy()

        rates = np.divide(
            100 * late,
            counts,
            out=np.zeros(len(counts), dtype=float),
            where=counts > 0
        )

        positions = x + offset

        ax.bar(
            positions,
            rates,
            width=width,
            color=color,
            label=label
        )

        for xpos, rate, n in zip(
            positions, rates, counts
        ):
            ax.text(
                xpos, rate + 0.16,
                f"{rate:.2f}%",
                ha="center",
                fontsize=8.5
            )

            ax.text(
                xpos, rate + 0.63,
                f"N={int(n):,}",
                ha="center",
                fontsize=7.5,
                color=COLORS["secondary"]
            )

    ax.set_ylim(0, 12)

    ax.set_xticks(x)
    ax.set_xticklabels(SLOTS, fontsize=8.5)

    ax.set_ylabel(
        "Late deliveries (% of orders)",
        fontsize=9
    )
    ax.set_xlabel(
        "Approval time slot",
        fontsize=9
    )

    ax.set_title(
        "Figure 2. Intra-state and inter-state "
        "late delivery rates by approval slot",
        fontsize=11,
        pad=12
    )

    ax.legend(
        loc="upper center",
        ncol=2,
        frameon=False,
        fontsize=8.5
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.text(
        0.5, 0.01,
        "Late rate (% of orders) by route type and approval slot. "
        "N = orders in each route and slot. "
        "Mixed-route orders (240) are not shown.",
        ha="center",
        fontsize=7
    )

    fig.tight_layout(rect=(0, 0.04, 1, 1))

    path = OUTPUT_DIR / "figure_02_revised.png"
    fig.savefig(path, dpi=200)

    display(fig)
    plt.close(fig)

    print(f"Saved: {path}")


plot_revised_figure_2(route_slots)

# Figure 3: Top 5 State Pairs

In [ ]:
def plot_revised_figure_3(
    pair_profiles,
    reference_rate
):

    pairs = (
        pair_profiles["pairs"]
        .sort_values(
            "late_orders",
            ascending=False
        )
        .head(5)
        .copy()
    )

    pairs["late_rate"] = (
        100 * pairs["late_orders"]
        / pairs["orders"]
    )

    labels = [
        f"{row.seller_state} → {row.customer_state} "
        f"(N={int(row.orders):,})"
        for row in pairs.itertuples()
    ]

    rates = pairs["late_rate"].to_numpy()
    y = np.arange(len(pairs))

    fig, ax = plt.subplots(figsize=(8.5, 4.2))

    ax.barh(
        y, rates,
        height=0.62,
        color=COLORS["green"]
    )

    ax.invert_yaxis()

    # Reference line
    ax.axvline(
        reference_rate,
        linestyle="--",
        linewidth=1,
        color=COLORS["reference"],
        label=(
            f"All inter-state orders: "
            f"{reference_rate:.2f}%"
        )
    )

    max_rate = rates.max()
    label_x = max_rate + 0.7

    for i, row in enumerate(pairs.itertuples()):
        ax.text(
            label_x, i,
            f"{row.late_rate:.2f}% "
            f"({int(row.late_orders):,} / "
            f"{int(row.orders):,})",
            va="center",
            fontsize=8.5
        )

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)

    ax.set_xlim(0, max_rate + 8)

    ax.set_xlabel(
        "Late deliveries (% of the pair's orders)",
        fontsize=9
    )

    ax.set_title(
        "Figure 3. Late rate by state pair",
        fontsize=11
    )

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.17),
        frameon=False,
        fontsize=8.5
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.text(
        0.5, 0.01,
        "Five inter-state pairs with the most late orders "
        "among orders approved 12:00–18:00, "
        "ranked by late count. "
        "Labels show late rate (late orders / total orders).",
        ha="center",
        fontsize=7
    )

    fig.tight_layout(rect=(0, 0.10, 1, 1))

    path = OUTPUT_DIR / "figure_03_revised.png"
    fig.savefig(path, dpi=200)

    display(fig)
    plt.close(fig)

    print(f"Saved: {path}")


plot_revised_figure_3(
    pair_profiles,
    inter_reference_rate
)

# Figure 4: Monthly Late Rate Comparison

In [ ]:
def plot_revised_figure_4(focus_time):

    monthly = focus_time["monthly"].copy()

    # Identify month column
    month_col = next(
        (
            col for col in ["month", "purchase_month"]
            if col in monthly.columns
        ),
        None
    )

    if month_col is None:
        raise ValueError(
            f"Month column not found: {monthly.columns.tolist()}"
        )

    # Identify scope column
    scope_col = next(
        (
            col for col in monthly.columns
            if monthly[col].astype(str)
            .str.contains("inter-state orders", case=False)
            .any()
        ),
        None
    )

    if scope_col is None:
        raise ValueError("Scope column not found.")

    # Normalize the scope labels
    monthly["_scope"] = (
        monthly[scope_col]
        .astype(str)
        .str.replace("→", "to", regex=False)
        .str.replace("->", "to", regex=False)
        .str.replace(r"[^a-zA-Z]", "", regex=True)
        .str.lower()
    )

    monthly["month_key"] = (
        monthly[month_col]
        .astype(str)
        .str.slice(0, 7)
    )

    months = (
        pd.period_range(
            "2017-01", "2018-08", freq="M"
        )
        .astype(str)
        .tolist()
    )

    # All inter-state orders
    all_inter = (
        monthly[
            monthly["_scope"].eq("allinterstateorders")
        ]
        .groupby("month_key")[["orders", "late_orders"]]
        .sum()
        .reindex(months, fill_value=0)
    )

    # SP -> RJ
    sp_rj = (
        monthly[
            monthly["_scope"].eq("sptorj")
        ]
        .groupby("month_key")[["orders", "late_orders"]]
        .sum()
        .reindex(months, fill_value=0)
    )

    if all_inter["orders"].sum() == 0:
        raise ValueError("All inter-state data not found.")

    if sp_rj["orders"].sum() == 0:
        raise ValueError("SP -> RJ data not found.")

    # Other inter-state = All inter-state - SP -> RJ
    other_orders = (
        all_inter["orders"] - sp_rj["orders"]
    )

    other_late = (
        all_inter["late_orders"] - sp_rj["late_orders"]
    )

    assert (other_orders >= 0).all()
    assert (other_late >= 0).all()

    # Calculate monthly late rates
    pair_rate = np.divide(
        100 * sp_rj["late_orders"].to_numpy(),
        sp_rj["orders"].to_numpy(),
        out=np.full(len(months), np.nan),
        where=sp_rj["orders"].to_numpy() > 0
    )

    other_rate = np.divide(
        100 * other_late.to_numpy(),
        other_orders.to_numpy(),
        out=np.full(len(months), np.nan),
        where=other_orders.to_numpy() > 0
    )

    x = np.arange(len(months))

    fig, ax = plt.subplots(figsize=(8.5, 4.4))

    # Highlight notable periods
    highlight_periods = [
        ("Nov–Dec 2017", "2017-11", "2017-12"),
        ("Feb–Mar 2018", "2018-02", "2018-03"),
    ]

    for label, start, end in highlight_periods:
        left = months.index(start) - 0.5
        right = months.index(end) + 0.5

        ax.axvspan(
            left, right,
            color=COLORS["highlight"],
            alpha=0.85,
            zorder=0
        )

        ax.text(
            (left + right) / 2,
            48,
            label,
            ha="center",
            va="top",
            fontsize=8,
            color=COLORS["secondary"]
        )

    # Other inter-state orders
    ax.plot(
        x, other_rate,
        marker="o",
        markersize=3.5,
        linewidth=1.6,
        color=COLORS["blue_line"],
        label="Other inter-state orders (excluding SP → RJ)"
    )

    # SP -> RJ
    ax.plot(
        x, pair_rate,
        marker="o",
        markersize=3.5,
        linewidth=1.6,
        color=COLORS["green_line"],
        label="SP → RJ"
    )

    ax.set_xlim(-0.5, len(months) - 0.5)
    ax.set_ylim(0, 50)

    ax.set_xticks(x[::2])
    ax.set_xticklabels(
        months[::2],
        rotation=45,
        ha="right",
        fontsize=8
    )

    ax.set_ylabel(
        "Late deliveries (% of monthly orders)",
        fontsize=9
    )

    ax.set_xlabel(
        "Purchase month",
        fontsize=9
    )

    ax.set_title(
        "Figure 4. Late rate by purchase month",
        fontsize=11
    )

    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=8
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.text(
        0.5, 0.01,
        "Late rate (% of monthly orders) by purchase month "
        "for orders approved 12:00–18:00. "
        "Shaded bands mark the periods discussed in the text.",
        ha="center",
        fontsize=7
    )

    fig.tight_layout(rect=(0, 0.04, 1, 1))

    path = OUTPUT_DIR / "figure_04_revised.png"
    fig.savefig(path, dpi=200)

    display(fig)
    plt.close(fig)

    print(f"Saved: {path}")


plot_revised_figure_4(focus_time)

# Verify Generated Figures

In [ ]:
figure_files = [
    "figure_01_revised.png",
    "figure_02_revised.png",
    "figure_03_revised.png",
    "figure_04_revised.png",
]

results = []

for filename in figure_files:
    path = OUTPUT_DIR / filename

    results.append({
        "Figure": filename,
        "Exists": path.exists(),
        "Size (KB)": (
            round(path.stat().st_size / 1024, 2)
            if path.exists() else None
        )
    })

verification = pd.DataFrame(results)

display(verification)

if verification["Exists"].all():
    print("SUCCESS: All 4 figures were exported.")
    print(f"Output directory: {OUTPUT_DIR}")
else:
    print("WARNING: Some figures are missing.")